# Darpa Problem Investigation #
This notebook is to investigate the DARPA Problem. In intrusion detection concerns the diffculty of detecting previously unseen attacks. Supervised models trained on known attack types fail to generalize to unfamiliar threats. This doesn't mean unsupervised models are much better as they face a problem as well. Unsupervised models may confuse legitimate anomalies with mailicious activity. This problem highlights the limits reliability of intrusion detection systems in network environments. 

I have used Random Forest and Logistic Regression which are usually goto supervised ML algos. I use the CICIDS2017 DDoS dataset to train the aforementioned models and test on the portscan dataset. 

In [3]:
%pip install scikit-learn matplotlib altair pandas numpy

import pandas as pd
import numpy as np

df = pd.read_csv("darpa\ddos.csv")
df

<>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Joshua Sanctus\AppData\Local\Temp\ipykernel_17800\1677404814.py:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df = pd.read_csv("darpa\ddos.csv")


Note: you may need to restart the kernel to use updated packages.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225740,61374,61,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
225741,61378,72,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
225742,61375,75,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
225743,61323,48,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [8]:
df.columns = df.columns.str.strip()
X = df.drop(columns=["Label"])
y = df["Label"]
print(X.shape)
print(y.shape)
print(y.value_counts())
print(y.unique())

(225745, 78)
(225745,)
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64
<ArrowStringArray>
['BENIGN', 'DDoS']
Length: 2, dtype: str


In [ ]:
missing = X.isnull().sum()
print(missing[missing > 0].sort_values(ascending=True))

numeric_X = X.select_dtypes(include=np.number)

print(X.duplicated().sum())
df.duplicated().sum()

infinite_counts = pd.Series(np.isinf(numeric_X).sum(),index=numeric_X.columns)
print(infinite_counts[infinite_counts > 0].sort_values(ascending=False))

Flow Bytes/s    4
dtype: int64
2633
Flow Packets/s    34
Flow Bytes/s      30
dtype: int64


In [16]:
X = X.replace([np.inf, -np.inf], np.nan)

missing = X.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print(X.isnull().sum().sum())
print(X.select_dtypes(exclude=np.number).columns.tolist())

Flow Bytes/s      34
Flow Packets/s    34
dtype: int64
68
[]


In [17]:
from sklearn.model_selection import train_test_split
y = y.map({"BENIGN": 0, "DDoS": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training class proportions: ", y_train.value_counts(normalize=True).round(3))

Training class proportions:  Label
1    0.567
0    0.433
Name: proportion, dtype: float64


In [19]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
def train_model(model, X_train, y_train, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler,", StandardScaler()))
    steps.append(("classfier,", model))
    pipeline = Pipeline(steps)
    pipeline.fit(X_train, y_train)

    return pipeline

In [21]:
def evaluate_model(model, X_test, y_test, model_name):
    predictions = model.predict(X_test)

    print(f"\n{'=' * 60}")
    print(f"Classification Report: {model_name}")
    print("=" * 60)

    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["BENIGN", "ATTACK"],
        digits=4,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))

    return predictions

In [23]:
import joblib 
import os
def export_model(model, model_name, export_dir="exported_models"):
    os.makedirs(export_dir, exist_ok=True)
    filepath = os.path.join(export_dir, f"{model_name}.pkl")
    joblib.dump(model, filepath)

    print(f"Exported {model_name} to: {filepath}")

    return filepath

In [25]:

models = {
    "random_forest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "scale": False
    },

    "logistic_regression": {
        "model": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ),
        "scale": True
    },

    "extra_trees": {
        "model": ExtraTreesClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "scale": False
    }
}

trained_models = {}
baseline_predictions = {}
exported_paths = {}

for name, config in models.items():
    fitted_model = train_model(
        config["model"],
        X_train,
        y_train,
        scale=config["scale"]
    )

    trained_models[name] = fitted_model

    baseline_predictions[name] = evaluate_model(
        fitted_model,
        X_test,
        y_test,
        name
    )

    exported_paths[name] = export_model(
        fitted_model,
        name
    )
print("As we can see the results are too perfect!!!")


Classification Report: random_forest
              precision    recall  f1-score   support

      BENIGN     0.9999    0.9999    0.9999     19544
      ATTACK     1.0000    1.0000    1.0000     25605

    accuracy                         1.0000     45149
   macro avg     1.0000    1.0000    1.0000     45149
weighted avg     1.0000    1.0000    1.0000     45149

Confusion Matrix:
[[19543     1]
 [    1 25604]]
Exported random_forest to: exported_models\random_forest.pkl

Classification Report: logistic_regression
              precision    recall  f1-score   support

      BENIGN     0.9988    0.9983    0.9985     19544
      ATTACK     0.9987    0.9991    0.9989     25605

    accuracy                         0.9987     45149
   macro avg     0.9987    0.9987    0.9987     45149
weighted avg     0.9987    0.9987    0.9987     45149

Confusion Matrix:
[[19511    33]
 [   24 25581]]
Exported logistic_regression to: exported_models\logistic_regression.pkl

Classification Report: extra_tr

Let's test the exported models on the portscan dataset

In [27]:
df_portscan = pd.read_csv("darpa\portscan.csv")
df_portscan

<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\Joshua Sanctus\AppData\Local\Temp\ipykernel_17800\3968090854.py:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  df_portscan = pd.read_csv("darpa\portscan.csv")


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286462,443,196135,49,57,1331,105841,570,0,27.163265,108.067176,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286463,443,378424,49,59,1325,104393,570,0,27.040816,108.095051,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286464,443,161800,70,103,1427,215903,570,0,20.385714,90.746389,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
286465,443,142864,50,62,1331,110185,570,0,26.620000,107.027727,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [28]:
df_portscan.columns = df_portscan.columns.str.strip()

print(df_portscan.shape)
print(df_portscan["Label"].value_counts())
print(df_portscan["Label"].unique())

(286467, 79)
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64
<ArrowStringArray>
['BENIGN', 'PortScan']
Length: 2, dtype: str


In [29]:
X_portscan = df_portscan.drop(columns=["Label"]).copy()

y_portscan = df_portscan["Label"].map({
    "BENIGN": 0,
    "PortScan": 1
})
if y_portscan.isna().any():
    unknown_labels = df_portscan.loc[
        y_portscan.isna(), "Label"
    ].unique()

    raise ValueError(f"Unexpected labels: {unknown_labels}")

training_columns = list(X_train.columns)
portscan_columns = list(X_portscan.columns)
missing_columns = sorted(set(training_columns) - set(portscan_columns))
extra_columns = sorted(set(portscan_columns) - set(training_columns))

print(missing_columns)
print(extra_columns)

if missing_columns or extra_columns:
    raise ValueError("Training and PortScan features do not match.")
X_portscan = X_portscan[training_columns]
X_portscan = X_portscan.replace([np.inf, -np.inf], np.nan)
print(y_portscan.value_counts())

[]
[]
Label
1    158930
0    127537
Name: count, dtype: int64


In [ ]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix


rf_loaded = joblib.load("exported_models/random_forest.pkl")
lr_loaded = joblib.load("exported_models/logistic_regression.pkl")
loaded_models = {
    "Random Forest": rf_loaded,
    "Logistic Regression": lr_loaded
}

portscan_predictions = {}
for model_name, model in loaded_models.items():
    predictions = model.predict(X_portscan)
    portscan_predictions[model_name] = predictions

    print(f"\n{'=' * 60}")
    print(model_name)
    print("=" * 60)

    print(classification_report(
        y_portscan,
        predictions,
        labels=[0, 1],
        target_names=["BENIGN", "PortScan"],
        digits=4,
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_portscan,
        predictions,
        labels=[0, 1]
    ))


PortScan Evaluation: Random Forest
              precision    recall  f1-score   support

      BENIGN     0.4453    0.9999    0.6162    127537
    PortScan     0.8235    0.0004    0.0007    158930

    accuracy                         0.4454    286467
   macro avg     0.6344    0.5001    0.3084    286467
weighted avg     0.6551    0.4454    0.2747    286467

Confusion Matrix:
[[127525     12]
 [158874     56]]

PortScan Evaluation: Logistic Regression
              precision    recall  f1-score   support

      BENIGN     0.4448    0.9983    0.6154    127537
    PortScan     0.0047    0.0000    0.0000    158930

    accuracy                         0.4445    286467
   macro avg     0.2248    0.4992    0.3077    286467
weighted avg     0.2006    0.4445    0.2740    286467

Confusion Matrix:
[[127326    211]
 [158929      1]]


In [ ]:
for model_name, predictions in portscan_predictions.items():
    print(f"\n{model_name}")
    print(pd.Series(predictions).value_counts().sort_index().rename(
        index={0: "BENIGN", 1: "ATTACK"}
    ))
    print("\nPredicted class percentages:")
    print(
        pd.Series(predictions)
        .map({0: "BENIGN", 1: "ATTACK"})
        .value_counts(normalize=True)
        .mul(100)
        .round(4)
    )


Random Forest
Predicted class counts:
BENIGN    286399
ATTACK        68
Name: count, dtype: int64

Predicted class percentages:
BENIGN    99.9763
ATTACK     0.0237
Name: proportion, dtype: float64

Logistic Regression
Predicted class counts:
BENIGN    286255
ATTACK       212
Name: count, dtype: int64

Predicted class percentages:
BENIGN    99.926
ATTACK     0.074
Name: proportion, dtype: float64
